# Day 089 | Settlement Prediction with ML

**Author:** Ripon Chandra Malo

## Capstone Project: Predicting Consolidation Settlement with ML

Consolidation settlement is one of the most critical design considerations in geotechnical engineering. Traditional methods based on Terzaghi's one-dimensional consolidation theory require careful determination of compression index (Cc), recompression index (Cr), and preconsolidation pressure -- parameters that can be difficult and time-consuming to obtain.

In this project, we train ML models to predict consolidation settlement from basic soil properties and compare the results with classical Terzaghi theory.

**Objectives:**
- Generate synthetic consolidation data (200 rows)
- Implement Terzaghi's classical settlement computation
- Train Random Forest, Gradient Boosting, and SVR models
- Compare ML predictions with theoretical values
- Analyze feature importance and model interpretability

In [ ]:
# ───── Part 1: Imports and Setup ─────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('All libraries loaded successfully.')

## Part 2: Synthetic Data Generation

We generate synthetic data based on the Terzaghi 1D consolidation settlement formula:

For normally consolidated clay (sigma_v0' < sigma_p'):
$$S = \frac{C_c \cdot H}{1 + e_0} \cdot \log_{10}\left(\frac{\sigma_{v0}' + \Delta\sigma}{\sigma_{v0}'}\right)$$

For overconsolidated clay where final stress exceeds preconsolidation pressure:
$$S = \frac{C_r \cdot H}{1 + e_0} \cdot \log_{10}\left(\frac{\sigma_p'}{\sigma_{v0}'}\right) + \frac{C_c \cdot H}{1 + e_0} \cdot \log_{10}\left(\frac{\sigma_{v0}' + \Delta\sigma}{\sigma_p'}\right)$$

In [ ]:
# ───── Part 2: Generate Synthetic Consolidation Dataset ─────
n_samples = 200

# Soil properties
layer_thickness = np.random.uniform(2.0, 15.0, n_samples)       # H (m)
void_ratio = np.random.uniform(0.5, 2.0, n_samples)             # e0
water_content = 30 + 20 * (void_ratio - 0.5) + np.random.normal(0, 5, n_samples)  # w (%)
water_content = np.clip(water_content, 15, 80)
liquid_limit = water_content + np.random.uniform(5, 25, n_samples)   # LL (%)
plastic_limit = water_content - np.random.uniform(5, 20, n_samples)  # PL (%)
plastic_limit = np.clip(plastic_limit, 10, 50)
plasticity_index = liquid_limit - plastic_limit                      # PI (%)

# Compression parameters (correlated with index properties)
Cc = 0.009 * (liquid_limit - 10) + np.random.normal(0, 0.03, n_samples)  # Skempton approx
Cc = np.clip(Cc, 0.05, 1.5)
Cr = Cc / np.random.uniform(4, 8, n_samples)  # Cr typically Cc/5 to Cc/7

# Stress conditions
depth_to_center = np.random.uniform(2.0, 20.0, n_samples)  # m
gamma = 16 + 2 * (1 - void_ratio / 2) + np.random.normal(0, 0.5, n_samples)  # kN/m3
gamma = np.clip(gamma, 14, 21)
sigma_v0 = gamma * depth_to_center  # kPa, effective vertical stress

# OCR and preconsolidation pressure
OCR = np.random.uniform(1.0, 4.0, n_samples)
sigma_p = sigma_v0 * OCR  # Preconsolidation pressure

# Applied stress increment
delta_sigma = np.random.uniform(20, 200, n_samples)  # kPa

# Compute Terzaghi settlement
settlement = np.zeros(n_samples)
for i in range(n_samples):
    sigma_final = sigma_v0[i] + delta_sigma[i]
    if sigma_final <= sigma_p[i]:  # Fully OC
        settlement[i] = (Cr[i] * layer_thickness[i] / (1 + void_ratio[i]) *
                        np.log10(sigma_final / sigma_v0[i]))
    else:  # Partially OC
        s_recomp = (Cr[i] * layer_thickness[i] / (1 + void_ratio[i]) *
                   np.log10(sigma_p[i] / sigma_v0[i]))
        s_virgin = (Cc[i] * layer_thickness[i] / (1 + void_ratio[i]) *
                   np.log10(sigma_final / sigma_p[i]))
        settlement[i] = s_recomp + s_virgin

# Add some noise to simulate measurement uncertainty
settlement_noisy = settlement * (1 + np.random.normal(0, 0.05, n_samples))
settlement_noisy = np.clip(settlement_noisy, 0.001, None)

# Convert to mm
settlement_mm = settlement_noisy * 1000
settlement_theory_mm = settlement * 1000

df = pd.DataFrame({
    'layer_thickness_m': np.round(layer_thickness, 2),
    'void_ratio': np.round(void_ratio, 3),
    'water_content_pct': np.round(water_content, 1),
    'liquid_limit': np.round(liquid_limit, 1),
    'plasticity_index': np.round(plasticity_index, 1),
    'unit_weight_kNm3': np.round(gamma, 2),
    'depth_to_center_m': np.round(depth_to_center, 2),
    'sigma_v0_kPa': np.round(sigma_v0, 1),
    'OCR': np.round(OCR, 2),
    'delta_sigma_kPa': np.round(delta_sigma, 1),
    'Cc': np.round(Cc, 4),
    'Cr': np.round(Cr, 4),
    'settlement_mm': np.round(settlement_mm, 2),
    'settlement_theory_mm': np.round(settlement_theory_mm, 2)
})

df.to_csv('settlement_data.csv', index=False)
print(f'Dataset shape: {df.shape}')
print(f'Settlement range: {df["settlement_mm"].min():.1f} - {df["settlement_mm"].max():.1f} mm')
df.head()

In [ ]:
# ───── Part 3: Exploratory Data Analysis ─────
print(df.describe().round(2))

In [ ]:
# ───── Part 4: Visualization ─────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].scatter(df['delta_sigma_kPa'], df['settlement_mm'], alpha=0.5,
                   c='steelblue', edgecolors='k', linewidth=0.3)
axes[0, 0].set_xlabel('Applied Stress (kPa)', fontsize=11)
axes[0, 0].set_ylabel('Settlement (mm)', fontsize=11)
axes[0, 0].set_title('(a) Stress vs Settlement', fontsize=12)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(df['Cc'], df['settlement_mm'], alpha=0.5,
                   c='coral', edgecolors='k', linewidth=0.3)
axes[0, 1].set_xlabel('Compression Index (Cc)', fontsize=11)
axes[0, 1].set_ylabel('Settlement (mm)', fontsize=11)
axes[0, 1].set_title('(b) Cc vs Settlement', fontsize=12)
axes[0, 1].grid(True, alpha=0.3)

sc = axes[0, 2].scatter(df['layer_thickness_m'], df['settlement_mm'],
                        c=df['OCR'], cmap='RdYlGn', alpha=0.6, edgecolors='k', linewidth=0.3)
plt.colorbar(sc, ax=axes[0, 2], label='OCR')
axes[0, 2].set_xlabel('Layer Thickness (m)', fontsize=11)
axes[0, 2].set_ylabel('Settlement (mm)', fontsize=11)
axes[0, 2].set_title('(c) Thickness vs Settlement', fontsize=12)
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].hist(df['settlement_mm'], bins=25, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Settlement (mm)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('(d) Settlement Distribution', fontsize=12)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].scatter(df['void_ratio'], df['Cc'], alpha=0.5,
                   c='forestgreen', edgecolors='k', linewidth=0.3)
axes[1, 1].set_xlabel('Void Ratio (e0)', fontsize=11)
axes[1, 1].set_ylabel('Cc', fontsize=11)
axes[1, 1].set_title('(e) Void Ratio vs Cc', fontsize=12)
axes[1, 1].grid(True, alpha=0.3)

# Correlation heatmap
corr_cols = ['layer_thickness_m', 'void_ratio', 'Cc', 'OCR', 'delta_sigma_kPa', 'settlement_mm']
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1, 2], square=True)
axes[1, 2].set_title('(f) Correlation Matrix', fontsize=12)

plt.tight_layout()
plt.savefig('eda_settlement.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 5: Data Preparation ─────
# Scenario 1: ML uses basic soil properties (no Cc, Cr - simulating field scenario)
features_basic = ['layer_thickness_m', 'void_ratio', 'water_content_pct',
                   'liquid_limit', 'plasticity_index', 'unit_weight_kNm3',
                   'depth_to_center_m', 'sigma_v0_kPa', 'OCR', 'delta_sigma_kPa']

# Scenario 2: ML uses all parameters (including Cc, Cr)
features_full = features_basic + ['Cc', 'Cr']

target = 'settlement_mm'

X_basic = df[features_basic].values
X_full = df[features_full].values
y = df[target].values

# Split (same indices for fair comparison)
X_b_train, X_b_test, y_train, y_test = train_test_split(
    X_basic, y, test_size=0.2, random_state=42)
X_f_train, X_f_test = train_test_split(
    X_full, test_size=0.2, random_state=42)[:2]

# Scale
scaler_b = StandardScaler()
X_b_train_sc = scaler_b.fit_transform(X_b_train)
X_b_test_sc = scaler_b.transform(X_b_test)

scaler_f = StandardScaler()
X_f_train_sc = scaler_f.fit_transform(X_f_train)
X_f_test_sc = scaler_f.transform(X_f_test)

print(f'Training: {X_b_train.shape[0]} samples')
print(f'Test:     {X_b_test.shape[0]} samples')
print(f'Basic features: {len(features_basic)}')
print(f'Full features:  {len(features_full)}')

In [ ]:
# ───── Part 6: Train ML Models ─────
models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12,
                                          min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                   learning_rate=0.05, random_state=42),
    'SVR': SVR(kernel='rbf', C=100, epsilon=0.1)
}

results = []

for name, model in models.items():
    # Train on basic features
    if name == 'SVR':
        model.fit(X_b_train_sc, y_train)
        y_pred = model.predict(X_b_test_sc)
    else:
        model.fit(X_b_train, y_train)
        y_pred = model.predict(X_b_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    results.append({'Model': f'{name} (basic)', 'R2': r2, 'RMSE_mm': rmse, 'MAE_mm': mae})
    print(f'{name} (basic) - R2: {r2:.4f}, RMSE: {rmse:.1f} mm, MAE: {mae:.1f} mm')

print()

In [ ]:
# ───── Part 7: Train with Full Features ─────
models_full = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=12,
                                          min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                   learning_rate=0.05, random_state=42),
    'SVR': SVR(kernel='rbf', C=100, epsilon=0.1)
}

preds_dict = {}

for name, model in models_full.items():
    if name == 'SVR':
        model.fit(X_f_train_sc, y_train)
        y_pred = model.predict(X_f_test_sc)
    else:
        model.fit(X_f_train, y_train)
        y_pred = model.predict(X_f_test)

    preds_dict[name] = y_pred
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    results.append({'Model': f'{name} (full)', 'R2': r2, 'RMSE_mm': rmse, 'MAE_mm': mae})
    print(f'{name} (full) - R2: {r2:.4f}, RMSE: {rmse:.1f} mm, MAE: {mae:.1f} mm')

In [ ]:
# ───── Part 8: Terzaghi Baseline ─────
# Get theoretical settlement for test samples
test_idx = df.index.isin(
    train_test_split(range(len(df)), test_size=0.2, random_state=42)[1]
)
y_terzaghi = df.loc[test_idx, 'settlement_theory_mm'].values

r2_terz = r2_score(y_test, y_terzaghi)
rmse_terz = np.sqrt(mean_squared_error(y_test, y_terzaghi))
mae_terz = mean_absolute_error(y_test, y_terzaghi)

results.append({'Model': 'Terzaghi Theory', 'R2': r2_terz, 'RMSE_mm': rmse_terz, 'MAE_mm': mae_terz})
print(f'Terzaghi Theory - R2: {r2_terz:.4f}, RMSE: {rmse_terz:.1f} mm, MAE: {mae_terz:.1f} mm')

In [ ]:
# ───── Part 9: Results Comparison ─────
results_df = pd.DataFrame(results).sort_values('R2', ascending=False).reset_index(drop=True)
print('\n=== Complete Model Comparison ===')
print(results_df.to_string(index=False))

In [ ]:
# ───── Part 10: Comparison Bar Chart ─────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Use full-feature models + Terzaghi for the chart
chart_df = results_df[results_df['Model'].str.contains('full|Terzaghi')].copy()
chart_df['Model'] = chart_df['Model'].str.replace(' (full)', '', regex=False)
bar_colors = ['#66BB6A', '#42A5F5', '#FFA726', '#EF5350'][:len(chart_df)]

axes[0].barh(chart_df['Model'], chart_df['R2'], color=bar_colors, edgecolor='black')
axes[0].set_xlabel('R-squared', fontsize=12)
axes[0].set_title('R-squared (higher = better)', fontsize=13)
for i, v in enumerate(chart_df['R2']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center')

axes[1].barh(chart_df['Model'], chart_df['RMSE_mm'], color=bar_colors, edgecolor='black')
axes[1].set_xlabel('RMSE (mm)', fontsize=12)
axes[1].set_title('RMSE (lower = better)', fontsize=13)
for i, v in enumerate(chart_df['RMSE_mm']):
    axes[1].text(v + 0.5, i, f'{v:.1f}', va='center')

axes[2].barh(chart_df['Model'], chart_df['MAE_mm'], color=bar_colors, edgecolor='black')
axes[2].set_xlabel('MAE (mm)', fontsize=12)
axes[2].set_title('MAE (lower = better)', fontsize=13)
for i, v in enumerate(chart_df['MAE_mm']):
    axes[2].text(v + 0.5, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 11: Predicted vs Observed ─────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, (name, y_pred) in enumerate(preds_dict.items()):
    vmin = min(y_test.min(), y_pred.min()) * 0.9
    vmax = max(y_test.max(), y_pred.max()) * 1.1
    axes[idx].scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidth=0.3, c='steelblue')
    axes[idx].plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=2, label='1:1 Line')
    axes[idx].set_xlabel('Observed Settlement (mm)', fontsize=11)
    axes[idx].set_ylabel('Predicted Settlement (mm)', fontsize=11)
    r2 = r2_score(y_test, y_pred)
    axes[idx].set_title(f'{name} (R2={r2:.3f})', fontsize=12)
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('pred_vs_obs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 12: Feature Importance ─────
rf_full = models_full['Random Forest']
gb_full = models_full['Gradient Boosting']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

rf_imp = pd.Series(rf_full.feature_importances_, index=features_full).sort_values()
rf_imp.plot(kind='barh', ax=axes[0], color='forestgreen', edgecolor='black')
axes[0].set_title('Random Forest Feature Importance', fontsize=13)
axes[0].set_xlabel('Importance', fontsize=11)

gb_imp = pd.Series(gb_full.feature_importances_, index=features_full).sort_values()
gb_imp.plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_title('Gradient Boosting Feature Importance', fontsize=13)
axes[1].set_xlabel('Importance', fontsize=11)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ───── Part 13: Cross-Validation ─────
print('=== 5-Fold Cross-Validation (Full Features) ===')
for name in ['Random Forest', 'Gradient Boosting']:
    if name == 'Random Forest':
        cv_model = RandomForestRegressor(n_estimators=200, max_depth=12,
                                        min_samples_leaf=3, random_state=42)
    else:
        cv_model = GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                            learning_rate=0.05, random_state=42)

    scores = cross_val_score(cv_model, X_full, y, cv=5, scoring='r2')
    print(f'{name}: R2 = {scores.mean():.4f} (+/- {scores.std():.4f})')

In [ ]:
# ───── Part 14: ML vs Terzaghi Comparison Plot ─────
best_pred = preds_dict['Gradient Boosting']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ML vs Observed
axes[0].scatter(y_test, best_pred, alpha=0.6, c='steelblue', edgecolors='k',
                linewidth=0.3, label='ML Prediction')
axes[0].scatter(y_test, y_terzaghi, alpha=0.6, c='coral', edgecolors='k',
                linewidth=0.3, marker='^', label='Terzaghi Theory')
vmin = min(y_test.min(), 0)
vmax = y_test.max() * 1.1
axes[0].plot([vmin, vmax], [vmin, vmax], 'k--', linewidth=1.5, label='1:1')
axes[0].set_xlabel('Observed Settlement (mm)', fontsize=12)
axes[0].set_ylabel('Predicted Settlement (mm)', fontsize=12)
axes[0].set_title('ML vs Terzaghi Predictions', fontsize=13)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error distribution
error_ml = y_test - best_pred
error_terz = y_test - y_terzaghi
axes[1].hist(error_ml, bins=20, alpha=0.6, color='steelblue', edgecolor='black', label='ML')
axes[1].hist(error_terz, bins=20, alpha=0.6, color='coral', edgecolor='black', label='Terzaghi')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Prediction Error (mm)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Error Distribution', fontsize=13)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ml_vs_terzaghi.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **ML models can predict consolidation settlement accurately**, achieving high R-squared values when provided with comprehensive soil properties.

2. **Compression index (Cc) and layer thickness are the dominant features**, consistent with the Terzaghi equation where settlement is directly proportional to both.

3. **Basic-features models (without Cc, Cr) still perform reasonably well**, demonstrating that ML can learn implicit compression behavior from index properties.

4. **Gradient Boosting and Random Forest outperform SVR** for this structured dataset, with Gradient Boosting showing slightly better generalization.

5. **ML complements classical methods** rather than replacing them -- theoretical understanding guides feature engineering while ML handles complex non-linear relationships.

## Next Steps

- Include secondary consolidation (creep) in the target variable
- Add time-rate prediction using time-series models
- Incorporate spatial variability and reliability analysis
- Test on real oedometer and field settlement monitoring data
- Develop confidence intervals for settlement predictions